# Spectral Analysis and Demodulation
Recovered implementation. Saved outputs and account metadata were removed for publication. Input data is not bundled.


In [ ]:
!pip install numpy scipy matplotlib

In [ ]:
import numpy as np
import scipy.io.wavfile as wav
import matplotlib.pyplot as plt
from IPython.display import Audio

In [ ]:
import os

os.makedirs('./Ex3_files', exist_ok=True)
from google.colab import files
import os

uploaded = files.upload()  # Choose your files from your computer

# Move uploaded files to the correct folder
os.makedirs('./Ex3_files', exist_ok=True)
for filename in uploaded.keys():
    os.rename(filename, f'./Ex3_files/{filename}')

In [ ]:
def plot_spectrum(filename):
    sr, data = wav.read(filename)

    if data.ndim > 1:
        data = data[:,0]  # Handle stereo by using one channel

    N = len(data)
    freqs = np.fft.rfftfreq(N, 1/sr)
    fft_spectrum = np.abs(np.fft.rfft(data))

    plt.figure(figsize=(10, 4))
    plt.plot(freqs, fft_spectrum)
    plt.title(f"Frequency Spectrum of {filename}")
    plt.xlabel("Frequency (Hz)")
    plt.ylabel("Amplitude")
    plt.xlim(0, sr//2)
    plt.grid()
    plt.show()


In [ ]:
folder_path = './Ex3_files/'
suspicious_file = None
ultrasonic_start = 19000
ultrasonic_end = 24000

print("Analyzing files for ultrasonic energy...\n")

for i in range(1, 5):
    filename = f'Ex3_sound{i}.wav'
    filepath = folder_path + filename

    sr, data = wav.read(filepath)
    if data.ndim > 1:
        data = data[:, 0]

    N = len(data)
    freqs = np.fft.rfftfreq(N, 1/sr)
    fft_spectrum = np.abs(np.fft.rfft(data))

    # Plot spectrum
    plt.figure(figsize=(10, 4))
    plt.plot(freqs, fft_spectrum)
    plt.title(f"Frequency Spectrum of {filename}")
    plt.xlabel("Frequency (Hz)")
    plt.ylabel("Amplitude")
    plt.xlim(0, sr//2)
    plt.grid()
    plt.show()

    # Detect ultrasonic energy
    ultrasonic_indices = np.where((freqs >= ultrasonic_start) & (freqs <= ultrasonic_end))
    ultrasonic_energy = np.sum(fft_spectrum[ultrasonic_indices])

    print(f"Total ultrasonic energy in {filename}: {ultrasonic_energy:.2f}")

    if suspicious_file is None or ultrasonic_energy > max_ultrasonic_energy:
        suspicious_file = filename
        max_ultrasonic_energy = ultrasonic_energy

print(f"\n⚠️ Suspicious activity likely detected in: {suspicious_file}")



In [ ]:
# Replace with the actual suspicious file you identified
filename = folder_path + 'Ex3_sound4.wav'

# Read the audio
sr, data = wav.read(filename)

# Handle stereo files
if data.ndim > 1:
    data = data[:, 0]

# Demodulation frequency - likely ultrasonic, so around 22kHz
mod_freq = 22050  # Half of common 44.1kHz sampling rate

# Generate carrier wave
t = np.arange(len(data)) / sr
carrier = np.cos(2 * np.pi * mod_freq * t)

# Multiply to shift hidden message to audible range
demodulated = data * carrier

# Normalize to avoid clipping
demodulated = demodulated / np.max(np.abs(demodulated))

# Save the recovered audio
output_path = folder_path + 'recovered_message.wav'
wav.write(output_path, sr, (demodulated * 32767).astype(np.int16))

# Listen to the recovered audio
Audio(output_path)


In [ ]:
test_frequencies = [19000, 20000, 21000, 22050, 23000, 24000]

sr, data = wav.read(filename)
if data.ndim > 1:
    data = data[:, 0]

t = np.arange(len(data)) / sr

for freq in test_frequencies:
    print(f"\n🔧 Testing demodulation frequency: {freq} Hz")

    carrier = np.cos(2 * np.pi * freq * t)
    demodulated = data * carrier
    demodulated = demodulated / np.max(np.abs(demodulated))  # Normalize

    output_path = folder_path + f'recovered_message_{freq}Hz.wav'
    wav.write(output_path, sr, (demodulated * 32767).astype(np.int16))

    display(Audio(output_path))
